In [1]:
from pathlib import Path
import sys
import json
import importlib

import numpy as np
import pandas as pd
import statsmodels.api as sm

cwd = Path.cwd()

if (cwd / "requirements.txt").exists():
    ROOT = cwd
elif (cwd.parent / "requirements.txt").exists():
    ROOT = cwd.parent
else:
    raise FileNotFoundError("Не найден корень проекта")

PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "reports" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.tr_vpin as tr_module
importlib.reload(tr_module)

with (ROOT / "research_config.json").open(
    encoding="utf-8"
) as file:
    config = json.load(file)

BASE_V = float(config["bucket_volume_btc"])
WINDOW = 50

classified = pd.read_csv(
    PROCESSED / "btcusdt_1min_classified_2020_q1.csv"
)

for column in ["minute_start", "available_at"]:
    classified[column] = pd.to_datetime(
        classified[column], utc=True
    )

classified["tick_sign"] = pd.to_numeric(
    classified["tick_sign"], errors="raise"
)

classified["is_classified"] = (
    classified["volume_btc"].gt(0)
    & classified["tick_sign"].isin([-1, 1])
)

train_base = pd.read_csv(PROCESSED / "forecast_train.csv")
test_base = pd.read_csv(PROCESSED / "forecast_test.csv")

train_base["sample"] = "train"
test_base["sample"] = "test"

panel = pd.concat(
    [train_base, test_base],
    ignore_index=True,
)

for column in ["timestamp", "target_end"]:
    panel[column] = pd.to_datetime(panel[column], utc=True)

panel = panel.sort_values("timestamp").reset_index(drop=True)

print("Базовый размер бакета, BTC:", BASE_V)

Базовый размер бакета, BTC: 1100.3497641153333


In [2]:
VARIANTS = {
    "V0p5": 0.5,
    "V1": 1.0,
    "V2": 2.0,
}

variant_columns = []
accounting_rows = []

source_volume = classified.loc[
    classified["is_classified"], "volume_btc"
].sum()

for label, multiplier in VARIANTS.items():
    volume = BASE_V * multiplier

    bucket_data = tr_module.build_volume_buckets(
        classified,
        bucket_volume=volume,
    )

    assert np.isclose(
        bucket_data["volume_btc"].sum(),
        source_volume,
        rtol=1e-10,
        atol=1e-8,
    )

    tr = tr_module.calculate_tr_vpin(
        bucket_data,
        bucket_volume=volume,
        window=WINDOW,
    )

    u2 = tr_module.calculate_u2_benchmark(
        classified,
        bucket_data,
        bucket_volume=volume,
        window=WINDOW,
    )

    variant = tr.merge(
        u2[["bucket_id", "u2_vpin"]],
        on="bucket_id",
        validate="one_to_one",
    )

    tr_name = f"tr_{label}"
    u2_name = f"u2_{label}"
    time_name = f"signal_time_{label}"

    signals = (
        variant
        .dropna(subset=["tr_vpin", "u2_vpin"])
        .sort_values(["available_at", "bucket_id"])
        .drop_duplicates("available_at", keep="last")
        [["available_at", "tr_vpin", "u2_vpin"]]
        .rename(columns={
            "available_at": time_name,
            "tr_vpin": tr_name,
            "u2_vpin": u2_name,
        })
    )

    panel = pd.merge_asof(
        panel.sort_values("timestamp"),
        signals.sort_values(time_name),
        left_on="timestamp",
        right_on=time_name,
        direction="backward",
        allow_exact_matches=True,
    )

    matched = panel[time_name].notna()

    assert (
        panel.loc[matched, time_name]
        <= panel.loc[matched, "timestamp"]
    ).all()

    variant_columns.extend([tr_name, u2_name])

    accounting_rows.append({
        "variant": label,
        "bucket_volume_btc": volume,
        "complete_buckets": len(tr),
        "valid_vpin_values": int(tr["tr_vpin"].notna().sum()),
    })

assert np.allclose(
    panel["tr_V1"],
    panel["tr_vpin"],
    equal_nan=True,
    rtol=1e-9,
    atol=1e-9,
)

assert np.allclose(
    panel["u2_V1"],
    panel["u2_vpin"],
    equal_nan=True,
    rtol=1e-9,
    atol=1e-9,
)

common = panel.dropna(subset=variant_columns).copy()

train = common.loc[common["sample"].eq("train")].copy()
test = common.loc[common["sample"].eq("test")].copy()

assert not train.empty
assert not test.empty
assert train["target_end"].max() <= test["timestamp"].min()

accounting_table = pd.DataFrame(accounting_rows)

print(accounting_table.round(6).to_string(index=False))
print("\nОбщая обучающая выборка:", len(train))
print("Общая тестовая выборка:", len(test))
print("Проверка базового размера бакета: OK")

variant  bucket_volume_btc  complete_buckets  valid_vpin_values
   V0p5         550.174882             12888              12839
     V1        1100.349764              6444               6395
     V2        2200.699528              3222               3173

Общая обучающая выборка: 1361
Общая тестовая выборка: 740
Проверка базового размера бакета: OK


In [3]:
TARGET = "rv_next_1h_pct"

controls = [
    "rv_prev_1h_pct",
    "log_volume_prev_1h",
]

specifications = {"Controls": controls}

for label in VARIANTS:
    specifications[f"Controls_TR_{label}"] = [
        *controls, f"tr_{label}"
    ]
    specifications[f"Controls_U2_{label}"] = [
        *controls, f"u2_{label}"
    ]

actual = test[TARGET].to_numpy(dtype=float)
rows = []

for name, features in specifications.items():
    X_train = sm.add_constant(
        train[features].astype(float),
        has_constant="add",
    )

    X_test = sm.add_constant(
        test[features].astype(float),
        has_constant="add",
    ).reindex(columns=X_train.columns)

    assert np.isfinite(X_train.to_numpy()).all()
    assert np.isfinite(X_test.to_numpy()).all()

    fit = sm.OLS(
        train[TARGET].astype(float),
        X_train,
        missing="raise",
    ).fit()

    forecast = np.asarray(fit.predict(X_test))
    error = forecast - actual

    rows.append({
        "model": name,
        "n_train": len(train),
        "n_test": len(test),
        "MSE": float(np.mean(error ** 2)),
        "RMSE": float(np.sqrt(np.mean(error ** 2))),
        "MAE": float(np.mean(np.abs(error))),
        "bias": float(np.mean(error)),
        "negative_predictions": int((forecast < 0).sum()),
    })

persistence_error = (
    test["rv_prev_1h_pct"].to_numpy(dtype=float) - actual
)

rows.append({
    "model": "Persistence",
    "n_train": len(train),
    "n_test": len(test),
    "MSE": float(np.mean(persistence_error ** 2)),
    "RMSE": float(np.sqrt(np.mean(persistence_error ** 2))),
    "MAE": float(np.mean(np.abs(persistence_error))),
    "bias": float(np.mean(persistence_error)),
    "negative_predictions": 0,
})

bucket_metrics = pd.DataFrame(rows).set_index("model")

reference_mse = bucket_metrics.loc["Controls", "MSE"]

bucket_metrics["MSE_gain_vs_controls_pct"] = (
    100 * (1 - bucket_metrics["MSE"] / reference_mse)
)

print(
    bucket_metrics
    .sort_values("RMSE")
    .round(6)
    .to_string()
)

bucket_metrics.to_csv(
    TABLES / "vpin_bucket_robustness.csv"
)

accounting_table.to_csv(
    TABLES / "vpin_bucket_variant_sizes.csv",
    index=False,
)

                  n_train  n_test       MSE      RMSE       MAE      bias  negative_predictions  MSE_gain_vs_controls_pct
model                                                                                                                    
Persistence          1361     740  1.423754  1.193212  0.487237  0.000281                     0                 18.289002
Controls_U2_V2       1361     740  1.675544  1.294428  0.536847 -0.455527                     0                  3.838449
Controls_U2_V1       1361     740  1.712479  1.308617  0.552514 -0.474192                     0                  1.718698
Controls_TR_V2       1361     740  1.721116  1.311913  0.557376 -0.482766                     0                  1.223012
Controls_TR_V1       1361     740  1.731604  1.315904  0.560959 -0.486483                     0                  0.621088
Controls_U2_V0p5     1361     740  1.736736  1.317853  0.562742 -0.486656                     0                  0.326551
Controls_TR_V0p5     136